# EnPreNiNet — Google Colab GPU Runner (Pilot & E0–E5 Ablation Matrix)

Thin orchestration notebook for reproducing **EnPreNiNet** (Ranjitha & Atham, 2026) on BDD100K night-vision data (3 classes: `car`, `person`, `traffic light`).

**Pre-training Engineering Guarantees:**
- **Fail-fast execution:** Every multi-step stage uses `%%bash` with `set -euo pipefail` so the pipeline stops immediately on any error.
- **Fast local SSD I/O:** Processed images (`data/processed`) live on local Colab SSD (`/content/work/processed`) to avoid Google Drive I/O throttling during training; `weights`, `runs`, and `artifacts` are persisted directly to Google Drive (`/content/drive/MyDrive/EnPreNiNet/`).
- **Pinned YOLOv5:** Locks `ultralytics/yolov5` to tag `v7.0` and logs both repo & YOLOv5 commit hashes to `artifacts/env_versions.txt`.

> **Runtime requirement:** Select `Runtime -> Change runtime type -> T4 / L4 / A100 GPU` before running.

## 1. Mount Google Drive, Clone Repository & Link Persistent Directories
- `weights/`, `runs/`, `artifacts/` -> Google Drive (`/content/drive/MyDrive/EnPreNiNet/`) for crash-safe persistence.
- `data/processed/` -> Local Colab SSD (`/content/work/processed`) for high-throughput training I/O.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -euo pipefail
mkdir -p /content/drive/MyDrive/EnPreNiNet/weights /content/drive/MyDrive/EnPreNiNet/runs /content/drive/MyDrive/EnPreNiNet/artifacts
mkdir -p /content/work/processed
if [ ! -d /content/enpreninet-night-detection ]; then
  git clone https://github.com/thiemdv/enpreninet-night-detection.git /content/enpreninet-night-detection
else
  git -C /content/enpreninet-night-detection pull --ff-only
fi
cd /content/enpreninet-night-detection
mkdir -p data
rm -rf data/processed weights runs artifacts
ln -s /content/work/processed data/processed
ln -s /content/drive/MyDrive/EnPreNiNet/weights weights
ln -s /content/drive/MyDrive/EnPreNiNet/runs runs
ln -s /content/drive/MyDrive/EnPreNiNet/artifacts artifacts
ls -la data/processed weights runs artifacts

In [ ]:
%cd /content/enpreninet-night-detection

## 2. Install Dependencies (Preserve Colab CUDA PyTorch), Pin YOLOv5 (`v7.0`) & Log Environment

In [ ]:
%%bash
set -euo pipefail
grep -Ev '^(torch|torchvision|#|\s*$)' requirements.txt | xargs -r pip install -q
if [ ! -d /content/yolov5 ]; then
  git clone https://github.com/ultralytics/yolov5.git /content/yolov5
fi
git -C /content/yolov5 fetch --tags
git -C /content/yolov5 checkout v7.0
grep -Ev '^(torch|torchvision|#|\s*$)' /content/yolov5/requirements.txt | xargs -r pip install -q
nvidia-smi > artifacts/env_versions.txt
python --version >> artifacts/env_versions.txt
python -c "import torch, torchvision, cv2, numpy; print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'torchvision:', torchvision.__version__, 'cv2:', cv2.__version__, 'numpy:', numpy.__version__)" >> artifacts/env_versions.txt
echo "enpreninet_commit=$(git rev-parse HEAD)" >> artifacts/env_versions.txt
echo "yolov5_commit=$(git -C /content/yolov5 rev-parse HEAD)" >> artifacts/env_versions.txt
cat artifacts/env_versions.txt

## 3. Prepare Raw BDD100K Dataset on Local SSD
Place `bdd100k_images_100k.zip` and `bdd100k_labels_release.zip` inside `/content/drive/MyDrive/EnPreNiNet/raw_zips/`, then unpack them to `/content/raw_bdd/`. Automatically verifies the extracted `bdd100k` root and labels JSON path.

In [ ]:
%%bash
set -euo pipefail
mkdir -p /content/raw_bdd
unzip -q -o /content/drive/MyDrive/EnPreNiNet/raw_zips/bdd100k_images_100k.zip -d /content/raw_bdd/
unzip -q -o /content/drive/MyDrive/EnPreNiNet/raw_zips/bdd100k_labels_release.zip -d /content/raw_bdd/
if [ -d /content/raw_bdd/bdd100k ]; then
  RAW_ROOT=/content/raw_bdd/bdd100k
else
  RAW_ROOT=/content/raw_bdd
fi
test -f "${RAW_ROOT}/labels/bdd100k_labels_images_train.json"
test -d "${RAW_ROOT}/images/100k/train"
echo "Verified BDD100K root: ${RAW_ROOT}"

## 4. Recommended GPU Pilot Experiment (200 Night Images, 2 Epochs)
Run this fast end-to-end pilot first (~10 min on T4 GPU) to verify real YOLOv5l, Faster R-CNN ResNet-152 GPU memory (`batch_size=2`), finite loss, E4/E5 fusion, and metrics before launching 50-epoch full training.

In [ ]:
%%bash
set -euo pipefail
RAW_ROOT=$( [ -d /content/raw_bdd/bdd100k ] && echo /content/raw_bdd/bdd100k || echo /content/raw_bdd )
python src/01_prepare_bdd.py --config configs/config.yaml --raw-dir "${RAW_ROOT}" --labels-json "${RAW_ROOT}/labels/bdd100k_labels_images_train.json" --out-dir /content/work/pilot_processed --max-images 200
python src/02_ghe_ensemble.py --config configs/config.yaml --data-root /content/work/pilot_processed --method standard_local_he
python src/03_train_infer_yolo.py --config configs/config.yaml --data-root /content/work/pilot_processed --image-variant standard_local_he --yolo-repo /content/yolov5 --epochs 2 --batch-size 8 --device cuda
python src/04_train_infer_frcnn.py --config configs/config.yaml --data-root /content/work/pilot_processed --image-variant standard_local_he --epochs 2 --batch-size 2 --device cuda
python src/05_fusion.py --config configs/config.yaml --image-variant standard_local_he --global-nms --output runs/fusion_standard_local_he/fusion_standard_local_he_pilot_test_preds.pt --device cuda
python src/06_evaluate.py --config configs/config.yaml --data-root /content/work/pilot_processed --eval-split test --preds Pilot_YOLO=runs/yolo_standard_local_he/yolo_standard_local_he_test_preds.pt Pilot_FRCNN=runs/frcnn_standard_local_he/frcnn_standard_local_he_test_preds.pt Pilot_EnPreNiNet=runs/fusion_standard_local_he/fusion_standard_local_he_pilot_test_preds.pt
cat artifacts/ablation_table.md

## 5. Full Dataset Preparation (`01_prepare_bdd.py`) & GHE Ensemble (`02_ghe_ensemble.py`)

In [ ]:
%%bash
set -euo pipefail
RAW_ROOT=$( [ -d /content/raw_bdd/bdd100k ] && echo /content/raw_bdd/bdd100k || echo /content/raw_bdd )
python src/01_prepare_bdd.py --config configs/config.yaml --raw-dir "${RAW_ROOT}" --labels-json "${RAW_ROOT}/labels/bdd100k_labels_images_train.json"
python src/02_ghe_ensemble.py --config configs/config.yaml --method standard_local_he
cp data/processed/dataset_meta.json artifacts/dataset_meta.json
cp data/processed/images/standard_local_he/ghe_meta.json artifacts/ghe_meta_standard_local_he.json

In [ ]:
%%bash
set -euo pipefail
# Optional extended GHE mode ablation: fig4_literal and fig4_normalized
python src/02_ghe_ensemble.py --config configs/config.yaml --method fig4_literal
python src/02_ghe_ensemble.py --config configs/config.yaml --method fig4_normalized

## 6. Full Dual-Branch GPU Training & Inference (`E0`, `E1`, `E2`, `E3`)
- **E0:** Baseline YOLOv5l on `raw` night frames
- **E1:** GHE-YOLOv5l on `standard_local_he` frames
- **E2:** Baseline Faster R-CNN ResNet-152 on `raw` night frames
- **E3:** GHE-Faster R-CNN ResNet-152 on `standard_local_he` frames

In [ ]:
%%bash
set -euo pipefail
# E0: Baseline YOLOv5l (raw)
python src/03_train_infer_yolo.py --config configs/config.yaml --image-variant raw --yolo-repo /content/yolov5 --device cuda
# E1: GHE-YOLOv5l (standard_local_he)
python src/03_train_infer_yolo.py --config configs/config.yaml --image-variant standard_local_he --yolo-repo /content/yolov5 --device cuda

In [ ]:
%%bash
set -euo pipefail
# E2: Baseline Faster R-CNN ResNet-152 (raw)
python src/04_train_infer_frcnn.py --config configs/config.yaml --image-variant raw --device cuda
# E3: GHE-Faster R-CNN ResNet-152 (standard_local_he)
python src/04_train_infer_frcnn.py --config configs/config.yaml --image-variant standard_local_he --device cuda

## 7. EnPreNiNet Score-Level Fusion (`E4` vs `E5`) & Full Evaluation (`06_evaluate.py`)
- **E4:** Fusion without Global NMS (`--no-global-nms` ablation)
- **E5:** Full EnPreNiNet Fusion with class-wise Global NMS (`--global-nms`)

In [ ]:
%%bash
set -euo pipefail
# E4: EnPreNiNet Fusion without Global NMS
python src/05_fusion.py --config configs/config.yaml --image-variant standard_local_he --no-global-nms --output runs/fusion_standard_local_he/fusion_standard_local_he_e4_noglobal_test_preds.pt --device cuda
# E5: EnPreNiNet Full Fusion with Global NMS
python src/05_fusion.py --config configs/config.yaml --image-variant standard_local_he --global-nms --output runs/fusion_standard_local_he/fusion_standard_local_he_e5_full_test_preds.pt --device cuda
# Evaluate E0-E5 against identical test split & dataset fingerprint
python src/06_evaluate.py --config configs/config.yaml --eval-split test --preds E0=runs/yolo_raw/yolo_raw_test_preds.pt E1=runs/yolo_standard_local_he/yolo_standard_local_he_test_preds.pt E2=runs/frcnn_raw/frcnn_raw_test_preds.pt E3=runs/frcnn_standard_local_he/frcnn_standard_local_he_test_preds.pt E4=runs/fusion_standard_local_he/fusion_standard_local_he_e4_noglobal_test_preds.pt E5=runs/fusion_standard_local_he/fusion_standard_local_he_e5_full_test_preds.pt
cat artifacts/ablation_table.md